# 02 · Cleaning, SQL storage and feature engineering
**Steps 5–7.** All cleaning logic lives in `src/cleaning.py`; each table has its own function with the reason for every choice in its docstring.

In [ ]:
import sys
sys.path.append("..")  # so `src` can be imported from the notebooks folder

import pandas as pd
pd.set_option("display.max_columns", 50)

from src.config import RAW_DIR
from src.data_loader import load_raw_tables
from src.cleaning import clean_all
from src.db import get_engine, reset_schema, load_tables, verify_counts, RAW_TABLES, FEATURE_TABLES
from src.features import build_all_features

## Step 5: Clean and preprocess
| Issue | Action |
| --- | --- |
| Duplicate rows / keys | dropped |
| Dates stored as text | converted to datetime |
| City names with accents / spaces | lower-case, accents removed |
| Zip codes as numbers | 5-character text (keeps leading zeros) |
| Misspelled columns (`lenght`) | renamed to `length` |
| Missing product category | `unknown` |
| Missing / zero product weight or size | category median |
| Category missing from translation | added with its own name |
| Payment type `not_defined`, 0 instalments | dropped / set to 1 |
| Several reviews for one order | latest review kept |
| Delivery date before purchase date | set to empty |
| Geolocation zip repeated many times | one row per zip (mean lat/lng) |
| Child rows with no parent | dropped so foreign keys hold |

In [ ]:
raw = load_raw_tables(RAW_DIR)
clean, clean_log = clean_all(raw)
clean_log

In [ ]:
clean["orders"].dtypes

## Step 6: Store the clean data in SQL
Credentials come from `.env` (never from the code). Tables are created from the ER diagram with primary and foreign keys, parents first.

In [ ]:
engine = get_engine()
reset_schema(engine)
load_tables(engine, clean, RAW_TABLES)

## Step 7: Feature engineering

In [ ]:
features = build_all_features(clean)
load_tables(engine, features, FEATURE_TABLES)
verify_counts(engine, {**clean, **features})

In [ ]:
features["order_features"][["order_id", "total_order_value", "delivery_days", "delivery_delay_days",
                            "delivery_status", "review_score", "customer_order_count", "is_repeat_customer"]].head()

In [ ]:
features["customer_features"].describe()

In [ ]:
features["seller_features"].sort_values("seller_revenue", ascending=False).head()